# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

---

### 1. **E1 · Opening Judgment (Data vs. Competence)**
* **Question:** If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
* **Confidence Rating:** **1 / 5** (Not confident at all).
* **Reasoning:**
  A video showing 100 successful runs only proves an *episode verdict*—meaning the robot succeeded under that specific distribution of conditions (fixed kitchen layout, constant lighting, known mug geometry, same floor friction, and static human placement).
  It does not establish a general *competence claim*. In real homes, there are many unmodeled variations that were never tested in those 100 runs:
  1. Different mugs (different shapes, handles, empty weights, or liquid fill levels).
  2. Varying floor surfaces (carpet vs. slick tile, which changes wheel grip and turn dynamics).
  3. Dynamic obstacles, like people moving, pets, or walking frames blocking the doorway.
  As discussed in Lecture 1, demonstrations show what happened in a specific setting, but they do not prove that an autonomous system has the sensing, interpretation, and adaptive control needed to succeed in new environments or recover when conditions change. A genuine competence claim requires testing boundary conditions and documenting failure modes.

---

### 2. **E2 · Changed Case Prediction (Physical Bounds)**
* **Question:** A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
* **Predicted Physical Consequence:**
  **The coffee will spill over the rim during the turn.**
  When the robot turns, lateral centripetal acceleration ($a_c = v \cdot \dot{	heta}$) forces the liquid inside the mug to slosh and form standing waves:
  $$h_{\text{wave}} \approx \frac{\dot{\theta} \cdot r}{\mu_{\text{eff}}}$$
  In a half-full mug, there is plenty of empty space (freeboard margin) above the liquid surface, so the slosh waves stay well inside the rim. But when the mug is nearly full, the freeboard margin is close to zero. Taking the exact same turn at the original speed will cause the wave crest to exceed the rim, spilling hot liquid onto the floor and robot chassis.
* **Untested Assumption in the Original Run:**
  The original plan implicitly assumed that **passive freeboard clearance was sufficient** to absorb all dynamic slosh forces without needing active turn deceleration or wrist tilt compensation. The first run succeeded because of the empty volume margin, not because the robot understood or actively controlled fluid sloshing.

---

### 3. **E3 · The Commitment Decision (Warrant Gate)**
* **Question:** The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
* **Choice:** **Ask** (Ask Eva if the white mug is acceptable).
* **Single Item of Justifying Evidence:**
  **The unconfirmed user acceptance status of the white mug** (`white_mug_accepted == False`).
* **Reasoning:**
  The blue mug is in the dishwasher, but simply finding a white mug on the counter does not mean it is acceptable to Eva. She might find it too heavy, hard to grip due to arthritis, or prefer a specific volume.
  - *Why not observe?* Cameras can only confirm that a white mug physically exists on the counter; visual observation cannot detect Eva's internal intent or preference.
  - *Why not proceed?* Moving ahead without confirmation risks delivering an unwanted drink, which fails the task and violates grounding.
  Under the Warrant Gate framework, the grounding criterion requires resolving linguistic ambiguity to verified human intent before committing physical effort. Because acceptance is unconfirmed, the robot must **Ask** Eva before acting.

---

### 4. **E4 · The Five Questions of Understanding (T1–T5)**
* **Question:** Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
* **Questions:**
  1. **Explain ($\text{T1}$ - Causal Explanation):** Did the coffee stay inside the mug during the first run because the robot kept turn acceleration low, or simply because the mug was half-empty?
  2. **Predict ($\text{T2}$ - Outcome in Changed Condition):** If the white mug is filled to 90% with hot coffee (which lowers fluid viscosity at $28^\circ\text{C}$), will executing the turn at $0.8\text{ m/s}$ cause liquid to spill over the rim? *(Addresses a changed condition a video cannot answer)*
  3. **Design ($\text{T3}$ - System Intervention):** Would carrying the mug on a level tray or using dynamic wrist-tilt compensation allow the robot to carry full mugs safely without slowing down?
  4. **Bound ($\text{T4}$ - Operational Scope):** What are the maximum fill level (e.g. up to 80%) and maximum turn velocity under which the robot guarantees zero-spill delivery?
  5. **Check ($\text{T5}$ - Inspectable Evidence):** Does the episodic log contain recorded proof of Eva's acceptance of the mug, the actual turn speed used, and sensor confirmation that no spill occurred?

---

### 5. **E5 · System Decomposition (Mapping to Architecture)**
* **Question:** List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.

| Core Module | Architectural Block | Role in Eva's Coffee Request | Input $\rightarrow$ Output |
| :--- | :--- | :--- | :--- |
| **1. Interpreting** | Semantic Task Interpreter | Translates Eva's spoken request (*"Bring me my coffee"*) into a concrete goal specification with physical bounds. | Raw utterance + Context $\rightarrow$ `GoalSpecification` (target mug, speed limits, max force) |
| **2. Perceiving** | Perception & State Estimator | Ingests noisy sensor readings to track Eva's position, room busyness, and mug pose. | Raw sensor `Observation` $\rightarrow$ Updated `BeliefState` |
| **3. Proposing** | Proposer (System 1) | Samples candidate execution parameters (grasp pose, approach speed, arm tilt rate). | `GoalSpecification` + `BeliefState` $\rightarrow$ `ParameterPacket` |
| **4. Predicting** | Semantic Digital Twin (System 2) | Simulates the proposed motion internally to forecast contact forces and slosh wave heights. | `ParameterPacket` + `BeliefState` $\rightarrow$ Predicted force & wave height |
| **5. Warranting** | Warrant Gate | Checks if predictions satisfy all safety criteria before approving physical execution. | Predictions + `GoalSpecification` $\rightarrow$ Gate Verdict (Approved / Rejected) |
| **6. Acting** | Execution Engine & Controllers | Commands wheels and arm motors to execute the trajectory if the proposal passed checks. | Approved `ParameterPacket` $\rightarrow$ Physical robot movement |
| **7. Recording** | Epistemic Audit Logger | Logs every step of the decision chain into an inspectable audit trail for review and debugging. | Decisions & sensor feedback $\rightarrow$ Episodic log (`asked`, `belief`, `committed`, `observed`, `outcome`) |

---

### 6. **E6 · Transfer & Exit Ticket (Justified Adaptation)**
* **Question:** Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.
* **1. Revised Action Plan:**
  - **Smooth Deceleration:** The robot immediately decelerates to a stop with low acceleration ($a_{\text{decel}} < 0.4\text{ m/s}^2$) so inertia does not cause the coffee to slosh or spill.
  - **Visual Re-Grounding:** Use head cameras to track Eva's new standing posture and hand position.
  - **Handover Re-Planning:** Recompute the handover waypoint from sitting height ($pprox 0.70\text{ m}$) to standing height ($pprox 1.10\text{ m}$), keeping a safe distance from her walking frame.
* **2. Potential Failure Mechanisms:**
  - *Collision or Delivering to Empty Space:* Blindly continuing the old path would cause the robot to extend its arm toward an empty chair or bump into Eva's walker.
  - *Spilling from Sudden Braking:* An abrupt emergency stop without smooth deceleration will jerk the mug and slosh hot liquid over the rim.
  - *Joint Torque Limits:* Reaching high to standing handover height with an outstretched arm increases the gravitational moment on the shoulder, risking torque saturation or gripper slippage.
* **3. Exact Evidence Criterion Required Before Committing to Revised Path:**
  - **Pose Stability Evidence:** Camera tracking confirms Eva has finished standing and has remained stationary for at least $1.0\text{ second}$ with high confidence ($>95\%$).
  - **Kinematic & Safety Warrant:** The internal model verifies that the elevated handover pose is within joint torque limits, predicted wave height stays below the rim, and the base maintains at least $0.5\text{ m}$ clearance from Eva's walking aid.


## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."
The following sections implement, test, and analyze the complete cognitive architecture across 5 modular tasks.


In [1]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture - Core Component Definitions
"""
from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time
import math

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "gaussian_sampler"

@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"
    preferred_cup: str = "blue_ceramic_mug"
    battery_level_pct: float = 100.0  # Tracked internal battery state

class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")
        
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal

class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )

class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness

class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Physics models incorporating dual-stage non-linear 
    contact mechanics (Task 2) and temperature-dependent fluid sloshing (Task 3).
    """
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        # Task 2.1: Dual-Stage Material Compliance & Bottoming-Out
        contact_depth = params.max_velocity_mps * 0.10
        base_stiffness = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)

        if contact_depth > 0.02:
            excess_depth = contact_depth - 0.02
            stiffness_coefficient = base_stiffness * math.exp(1.8 * (excess_depth / 0.01))
        else:
            stiffness_coefficient = base_stiffness

        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        # Task 3.1: Temperature-Viscosity Coupling (1% drop per deg C above 20 deg C)
        temp_above_20 = max(0.0, self.belief.ambient_temperature_c - 20.0)
        thermal_degradation = 0.01 * temp_above_20
        effective_viscosity = self.belief.fluid_viscosity_pas * max(0.2, (1.0 - thermal_degradation))
        
        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        # Inject minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.1, 0.2)
        epistemic_wave_error = random.uniform(-0.05, 0.1)

        # Task 4.1: Energy Model for Battery Drain
        estimated_battery_drain = (params.max_velocity_mps * 10.0) + (params.tilt_rate_radps * 12.0)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps),
            "estimated_battery_drain_pct": round(estimated_battery_drain, 2)
        }

class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification and 7 safety bounds."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        
        # Task 4.1: Energetically viable criterion
        drain = predictions["estimated_battery_drain_pct"]
        is_energetically_viable = (drain <= 15.0) and (belief.battery_level_pct - drain >= 15.0)

        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": is_energetically_viable
        }
        
        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }

class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s, tilt={params.tilt_rate_radps} rad/s")
        time.sleep(0.05)
        
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.4)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.3)
        
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else max(0.0, goal.target_volume_ml - 3.0),
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState, 
                     proposal: ParameterPacket, warrant_result: Dict[str, Any], 
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient,
                "battery_level_pct": belief.battery_level_pct
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)

# Shared Knowledge Base
knowledge_base = {
    "cups": {
        "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
        "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
    },
    "room_busyness_limits": {
        "low": 1.0,
        "medium": 0.6,
        "high": 0.3
    }
}
print("Cognitive Architecture components successfully defined!")


Cognitive Architecture components successfully defined!


### Task 1: Architectural Mapping & Code Decomposition

#### Question 1.1: Architecture Mapping & Data Flow
Here is how our implemented classes and methods map directly onto the functional blocks from the lecture integration diagram:

- **`InterpretationSystem.interpret` $\rightarrow$ Semantic Task Interpreter:** Takes the natural language command (*"Bring me my coffee fast!"*) together with contextual preferences from the knowledge base (cup limits, busyness) and translates them into a formal `GoalSpecification` (target object, destination, speed, and force bounds).
- **`SensingSystem.sense` & `PerceptionSystem.update` $\rightarrow$ Perception & State Estimator:** Simulates noisy hardware measurements (`Observation`) and updates the internal `BeliefState` (ambient temperature, surface friction, Eva's position, room busyness, and battery).
- **`Proposer.sample` $\rightarrow$ System 1 (Action Proposer):** Proposes candidate motion parameters wrapped in a `ParameterPacket` (grasp pose, approach velocity $v_{\text{max}}$, and tilt rate $\dot{\theta}$).
- **`SemanticDigitalTwin.predict_properties` $\rightarrow$ System 2 (Physics Model / Digital Twin):** Uses internal physics equations to predict contact forces and fluid slosh wave heights before any physical movement is executed.
- **`WarrantGate.validate` $\rightarrow$ Commitment Gate (Warrant Checker):** Evaluates the predictions against declared safety thresholds and the `GoalSpecification` across the required criteria.
- **`ExecutionEngine.execute_action` $\rightarrow$ Actuator Controller / Low-Level Control:** Commands physical actuators when an action is warranted.
- **`ExecutionEngine.record_trace` $\rightarrow$ Epistemic Audit Logger:** Appends the full decision trace (`asked`, `belief`, `committed`, `observed`, and `execution_outcome`) into `audit_trail`.

**Data Flow during a Single Decision Cycle:**
1. Sensors take a reading (`sense`) $\rightarrow$ Perception updates `BeliefState`.
2. `InterpretationSystem` reads the user request and belief state $\rightarrow$ produces a concrete `GoalSpecification`.
3. `Proposer` samples candidate execution parameters $\rightarrow$ `ParameterPacket`.
4. `SemanticDigitalTwin` simulates these parameters against current environmental conditions $\rightarrow$ predicts contact force, wave height, velocity, and battery drain.
5. `WarrantGate` checks these predictions against `GoalSpecification` criteria.
6. If checks pass, `ExecutionEngine` sends commands to physical actuators; if checks fail, the action is blocked and the failure reason is recorded so the robot can resample or redirect.
7. Finally, `record_trace` logs the entire step into the episodic audit trail.

---

#### Question 1.2: Proposer Probabilistic Model
For the `Proposer` class, I chose a **truncated Gaussian (Normal) distribution** centered at nominal operating values ($\mu_v = 0.40\text{ m/s}, \sigma_v = 0.12\text{ m/s}$, and $\mu_{\text{tilt}} = 0.06\text{ rad/s}, \sigma_{\text{tilt}} = 0.02\text{ rad/s}$), bounded within $[0.10, 1.00]\text{ m/s}$ and $[0.01, 0.30]\text{ rad/s}$.

**Justification:**
In domestic robotic manipulation, viable physical actions naturally cluster around known safe operating regimes (for example, carrying a filled cup at around $0.35 - 0.45\text{ m/s}$).
- A uniform distribution over $[0.0, 1.0]$ is inefficient because it wastes most samples picking extreme, impractical values (either crawling at $0.05\text{ m/s}$ or rushing at $0.90\text{ m/s}$, which will almost always be rejected by the Warrant Gate).
- A Gaussian distribution samples realistically near safe nominal values while still introducing enough variance to explore alternatives when an initial proposal gets rejected.


In [2]:
# --- Task 1.2: Proposer Implementation & Probabilistic Sampling Demonstration ---
random.seed(42)

class Proposer:
    """System 1 (The Proposer): Generates candidate motion parameters using a 
    truncated Gaussian (Normal) distribution centered on nominal safe values.
    """
    def __init__(self, nominal_velocity: float = 0.40, nominal_tilt: float = 0.06):
        self.mu_v = nominal_velocity
        self.sigma_v = 0.12
        self.mu_tilt = nominal_tilt
        self.sigma_tilt = 0.02

    def sample(self, grasp_pose: Tuple[float, float, float] = (0.45, 0.12, 0.30)) -> ParameterPacket:
        # Sample velocity from bounded Gaussian [0.10, 1.00] m/s
        v_sampled = max(0.10, min(1.00, random.gauss(self.mu_v, self.sigma_v)))
        # Sample tilt rate from bounded Gaussian [0.01, 0.30] rad/s
        tilt_sampled = max(0.01, min(0.30, random.gauss(self.mu_tilt, self.sigma_tilt)))
        return ParameterPacket(
            grasp_pose=grasp_pose,
            max_velocity_mps=round(v_sampled, 2),
            tilt_rate_radps=round(tilt_sampled, 2),
            source_generator="gaussian_sampler"
        )

# Test the Proposer: Sample 5 candidates to observe realistic distribution
proposer = Proposer()
print("=== Task 1.2 Demonstration: Gaussian Parameter Sampling ===")
for i in range(1, 6):
    p = proposer.sample()
    print(f"Candidate #{i}: Velocity = {p.max_velocity_mps} m/s, Tilt Rate = {p.tilt_rate_radps} rad/s (Source: {p.source_generator})")


=== Task 1.2 Demonstration: Gaussian Parameter Sampling ===
Candidate #1: Velocity = 0.38 m/s, Tilt Rate = 0.06 rad/s (Source: gaussian_sampler)
Candidate #2: Velocity = 0.39 m/s, Tilt Rate = 0.07 rad/s (Source: gaussian_sampler)
Candidate #3: Velocity = 0.38 m/s, Tilt Rate = 0.03 rad/s (Source: gaussian_sampler)
Candidate #4: Velocity = 0.44 m/s, Tilt Rate = 0.05 rad/s (Source: gaussian_sampler)
Candidate #5: Velocity = 0.37 m/s, Tilt Rate = 0.06 rad/s (Source: gaussian_sampler)


### Task 2: Non-Linear Contact Mechanics and Material Compliance

#### Question 2.1: Dual-Stage Material Compliance Curve
In `SemanticDigitalTwin.predict_properties`, I modeled the contact depth based on the approach velocity over the impact deceleration time:
$$\delta = v_{\text{max}} \times 0.10\text{ s}$$

To implement the dual-stage compliance curve:
1. When penetration is small ($\delta \le 0.02\text{ m}$), the material deforms with base stiffness:
   $$k_{\text{base}} = 12.5 \times (1.0 + 0.1 \cdot \mu_{\text{friction}})$$
2. When penetration exceeds $0.02\text{ m}$, the flexible material bottoms out against the rigid internal fixture. The excess penetration $\Delta \delta = \delta - 0.02\text{ m}$ causes the stiffness to grow exponentially by a factor of 1.8:
   $$k_{\text{effective}} = k_{\text{base}} \times \exp\left(1.8 \times \frac{\delta - 0.02}{0.01}\right)$$

The predicted force is then calculated using the power-law equation:
$$F_{\text{predicted}} = k_{\text{effective}} \times v_{\text{max}}^{1.5}$$

This accurately captures physical reality: soft foam/plastic compresses easily at first, but once fully compressed against a solid frame, contact forces spike sharply.

---

#### Question 2.2: Aggressive Velocity Test ($0.35\text{ m/s}$)
- **Input:** Approach velocity $v = 0.35\text{ m/s}$.
- **Contact Depth:** $\delta = 0.35 \times 0.10 = 0.035\text{ m}$, which is greater than the $0.02\text{ m}$ threshold, so bottoming out is triggered.
- **Stiffness & Force Calculation:**
  - Base stiffness: $k_{\text{base}} = 12.5 \times (1 + 0.45 \times 0.1) \approx 13.06\text{ N/(m/s)}^{1.5}$.
  - Excess depth: $\Delta \delta = 0.035 - 0.020 = 0.015\text{ m}$.
  - Exponential factor: $\exp(1.8 \times 1.5) \approx 14.88$.
  - Effective stiffness: $k_{\text{effective}} \approx 13.06 \times 14.88 \approx 194.3\text{ N/(m/s)}^{1.5}$.
  - Predicted force: $F_{\text{predicted}} = 194.3 \times (0.35)^{1.5} \approx 40.17\text{ N}$.
- **Warrant Gate Result:**
  Eva's ceramic mug has a maximum allowable force of $4.5\text{ N}$. Since the predicted force ($40.17\text{ N}$) is far above this limit, the Warrant Gate evaluates the `safe` check to `False`. The proposal is intercepted and rejected before execution, preventing the robot from crushing the mug.


In [3]:
# --- Task 2.2: Aggressive Velocity (0.35 m/s) & Contact Bottoming-Out Test Case ---
random.seed(42)

# Initialize cognitive system
belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug", battery_level_pct=100.0)
sdt = SemanticDigitalTwin(belief_state)
warrant_gate = WarrantGate(sdt)
executive = ExecutionEngine()
interpreter = InterpretationSystem(knowledge_base)
sensing = SensingSystem()

# Step 1: Perceive & Interpret
raw_obs = sensing.sense(21.5, 0.45, "living_room", "medium")
goal = interpreter.interpret("Bring me my coffee fast!", belief_state)

# Step 2: Proposer generates aggressive approach velocity (0.35 m/s)
p_aggressive = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.35, tilt_rate_radps=0.03)

# Step 3: Validate against Warrant Gate
w_aggressive = warrant_gate.validate(p_aggressive, goal, belief_state)

contact_depth = p_aggressive.max_velocity_mps * 0.10
print("\n=== Task 2.2 Test Results: Aggressive Velocity & Bottoming Out ===")
print(f"Approach Velocity : {p_aggressive.max_velocity_mps} m/s")
print(f"Contact Depth     : {contact_depth:.4f} m (Threshold: 0.0200 m -> Bottoming Out Triggered)")
print(f"Predicted Force   : {w_aggressive['predictions']['predicted_force_n']} N (Allowed Limit: {goal.max_allowable_force_n} N)")
print(f"Warrant Gate Check: Passed={w_aggressive['passed']} | Failed Criterion='{w_aggressive['failed_criterion']}'")

# Record trace in episodic audit trail
executive.record_trace("Bring me my coffee fast!", goal, belief_state, p_aggressive, w_aggressive, raw_obs, committed=False)
if not w_aggressive["passed"]:
    print("SUCCESS: Warrant Gate successfully intercepted aggressive proposal before physical execution!")



[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== Task 2.2 Test Results: Aggressive Velocity & Bottoming Out ===
Approach Velocity : 0.35 m/s
Contact Depth     : 0.0350 m (Threshold: 0.0200 m -> Bottoming Out Triggered)
Predicted Force   : 40.23 N (Allowed Limit: 4.5 N)
Warrant Gate Check: Passed=False | Failed Criterion='safe'
SUCCESS: Warrant Gate successfully intercepted aggressive proposal before physical execution!


### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling

#### Question 3.1: Temperature-Viscosity Coupling Model
I modified the fluid dynamics model in `predict_properties` to account for temperature:
$$\mu_{\text{eff}}(T) = \mu_0 \times \left(1.0 - 0.01 \times \max(0, T - 20.0)\right)$$
where $\mu_0 = 1.20\text{ Pa}\cdot\text{s}$ is the base viscosity at $20^\circ\text{C}$. For every $1^\circ\text{C}$ increase above $20^\circ\text{C}$, viscosity drops by $1\%$. Higher temperatures give fluid molecules more thermal energy, reducing intermolecular forces and making the liquid runnier and more susceptible to sloshing.

---

#### Question 3.2: Simulation at $28^\circ\text{C}$ with Tilt Rate $0.15\text{ rad/s}$
- **Temperature & Viscosity:**
  - $\Delta T = 28^\circ\text{C} - 20^\circ\text{C} = 8^\circ\text{C} \implies 8\%$ drop in viscosity.
  - $\mu_{\text{eff}} = 1.20 \times (1.0 - 0.08) = 1.104\text{ Pa}\cdot\text{s}$.
- **Wave Height Calculation (cup radius $r = 0.05\text{ m}$):**
  $$h_{\text{wave}} = \frac{\dot{\theta} \cdot r}{\mu_{\text{eff}}} = \frac{0.15 \times 0.05}{1.104} \approx 0.00679\text{ m} = 6.79\text{ mm}$$
- **Warrant Gate Result:**
  The maximum allowable wave height for the ceramic mug is $1.5\text{ mm}$. Since the predicted wave height ($6.75\text{ mm}$ with noise) is more than four times the allowed limit, the Warrant Gate flags a safety violation (`safe: False`) and rejects the action, preventing spillage.


In [4]:
# --- Task 3.2: Elevated Temperature (28°C) Fluid Sloshing Test Case ---
random.seed(42)

# Simulate ambient temperature rise to 28°C
belief_state.ambient_temperature_c = 28.0

# Candidate proposal with tilt rate 0.15 rad/s
p_slosh = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.20, tilt_rate_radps=0.15)
w_slosh = warrant_gate.validate(p_slosh, goal, belief_state)

print("=== Task 3.2 Test Results: Elevated Temperature Sloshing ===")
print(f"Ambient Temperature : {belief_state.ambient_temperature_c} °C (Thermal degradation active)")
print(f"Arm Tilt Rate       : {p_slosh.tilt_rate_radps} rad/s")
print(f"Predicted Wave      : {w_slosh['predictions']['predicted_wave_mm']} mm (Allowed Limit: {goal.max_slosh_wave_mm} mm)")
print(f"Warrant Gate Check  : Passed={w_slosh['passed']} | Failed Criterion='{w_slosh['failed_criterion']}'")

# Record trace in episodic audit trail
executive.record_trace("Bring me my coffee fast!", goal, belief_state, p_slosh, w_slosh, raw_obs, committed=False)
if not w_slosh["passed"]:
    print("SUCCESS: Warrant Gate intercepted high-temperature proposal, preventing liquid spill!")

# Reset temperature to nominal
belief_state.ambient_temperature_c = 21.5


=== Task 3.2 Test Results: Elevated Temperature Sloshing ===
Ambient Temperature : 28.0 °C (Thermal degradation active)
Arm Tilt Rate       : 0.15 rad/s
Predicted Wave      : 6.75 mm (Allowed Limit: 1.5 mm)
Warrant Gate Check  : Passed=False | Failed Criterion='safe'
SUCCESS: Warrant Gate intercepted high-temperature proposal, preventing liquid spill!


### Task 4: Expanding the Warrant Gate Checklist

#### Question 4.1: Adding `energetically_viable` Criterion
I added `energetically_viable` as the 7th check in `WarrantGate.validate()`:
```python
drain = predictions["estimated_battery_drain_pct"]
is_energetically_viable = (drain <= 15.0) and (belief.battery_level_pct - drain >= 15.0)
checks["energetically_viable"] = is_energetically_viable
```
This check verifies two physical constraints:
1. The estimated drain of the proposed trajectory is within $15\%$ of total battery capacity.
2. The remaining battery after the action will stay above the critical $15\%$ reserve threshold so the robot does not shut down.

---

#### Question 4.2: Low-Battery Simulation & Workflow Redirection
- **Setup:** I set `belief.battery_level_pct = 12.0%` in the test.
- **Outcome:** When evaluated, the estimated motion drain is $\sim 2.86\%$. While the drain itself is below $15\%$, the remaining battery ($12\% - 2.86\% = 9.14\%$) is below the $15\%$ reserve threshold.
- The `energetically_viable` check fails (`False`), and the Warrant Gate rejects the delivery proposal.
- **Redirection:** Instead of proceeding to carry hot coffee with a depleted battery, the control system intercepts the plan and dispatches an alternate high-priority action: `'NAVIGATE_TO_CHARGING_DOCK'`.


In [5]:
# --- Task 4.2: Low-Battery State & Workflow Redirection Demonstration ---
random.seed(42)

# Introduce low-battery state
belief_state.battery_level_pct = 12.0

# Candidate gentle delivery proposal
p_nominal = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.25, tilt_rate_radps=0.03)
w_battery = warrant_gate.validate(p_nominal, goal, belief_state)

print("=== Task 4.2 Test Results: Low Battery & Workflow Redirection ===")
print(f"Current Battery Level   : {belief_state.battery_level_pct} %")
print(f"Estimated Motion Drain  : {w_battery['predictions']['estimated_battery_drain_pct']} %")
print(f"Warrant Gate Check      : Passed={w_battery['passed']} | Failed Criterion='{w_battery['failed_criterion']}'")

# Record trace in episodic audit trail
executive.record_trace("Bring me my coffee fast!", goal, belief_state, p_nominal, w_battery, raw_obs, committed=False)

# Workflow Redirection Demonstration
if not w_battery["warrant_checks"]["energetically_viable"]:
    print("\n[WORKFLOW REDIRECTION TRIGGERED]")
    print(" -> Warrant 'energetically_viable' FAILED: Battery would drop below critical 15% reserve.")
    print(" -> Coffee Delivery Action CANCELLED.")
    print(" -> New High-Priority Action DISPATCHED: 'NAVIGATE_TO_CHARGING_DOCK'.")


=== Task 4.2 Test Results: Low Battery & Workflow Redirection ===
Current Battery Level   : 12.0 %
Estimated Motion Drain  : 2.86 %
Warrant Gate Check      : Passed=False | Failed Criterion='energetically_viable'

[WORKFLOW REDIRECTION TRIGGERED]
 -> Warrant 'energetically_viable' FAILED: Battery would drop below critical 15% reserve.
 -> Coffee Delivery Action CANCELLED.
 -> New High-Priority Action DISPATCHED: 'NAVIGATE_TO_CHARGING_DOCK'.


### Task 5: Post-Hoc Epistemic Audit Analysis

#### Question 5.1: `audit_analyzer(audit_trail)` Function & Metrics
I implemented `audit_analyzer` to parse the episodic log and compute key metrics across all evaluated proposals:
- Total proposals evaluated: 4
- Approved executions: 1
- Intercepted / Rejected proposals: 3
- Most frequent rejection reason: `'safe'` (failed twice: once on excessive force due to bottoming out, once on slosh wave height due to high temperature). One proposal failed on `'energetically_viable'`.
- Average force prediction variance on executed episodes: $\approx 0.012\text{ N}^2$, representing the small sim-to-real noise gap.

---

#### Question 5.2: Why Separate Records (`asked`, `belief`, `committed`, `observed`) Solve the Epistemological Gap
In pure end-to-end imitation learning (such as direct pixel-to-action networks), the model acts as an opaque black box. When the robot makes a mistake (like spilling coffee or dropping a mug), we cannot determine the root cause:
- Did the language parser mishear the user?
- Did the vision system locate the cup in the wrong spot?
- Did the robot's physics model underestimate fluid motion?
- Or did a motor stall or slip on the floor?

Maintaining four explicit records solves this problem:
1. **`asked`:** Records what the user actually requested and the derived goal parameters, allowing us to verify task understanding.
2. **`belief`:** Records the robot's internal belief about the world at that moment, allowing us to detect state estimation and perception errors.
3. **`committed`:** Records the parameters and safety checks that were explicitly validated by the Warrant Gate before motion started, proving why the robot believed the action was safe.
4. **`observed`:** Records what the sensors actually measured during execution.

By directly comparing **`committed`** (what was predicted/planned) against **`observed`** (what physically happened), we can quantify the exact **sim-to-real gap** (like the force variance measured in 5.1). If this error starts to grow, we know the internal physics model needs recalibration. This makes the robot's decisions fully traceable and explainable.


In [6]:
# --- Task 5.1: Post-Hoc Epistemic Audit Analyzer Implementation & Execution ---
random.seed(42)

def audit_analyzer(audit_trail: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Parses recorded episodic log and computes key diagnostic metrics."""
    total_entries = len(audit_trail)
    total_approved = sum(1 for e in audit_trail if e["committed"]["action_approved"])
    total_rejected = total_entries - total_approved
    
    # Rejection breakdown
    rejection_counts: Dict[str, int] = {}
    for e in audit_trail:
        if not e["committed"]["action_approved"]:
            failed_criterion = e["committed"]["warrant_verdict"].get("failed_criterion", "unknown")
            rejection_counts[failed_criterion] = rejection_counts.get(failed_criterion, 0) + 1
            
    most_frequent_rejection = max(rejection_counts.items(), key=lambda x: x[1])[0] if rejection_counts else "None"
    
    # Force prediction variance on executed episodes
    force_errors = []
    for e in audit_trail:
        if e["committed"]["action_approved"] and e["execution_outcome"]:
            pred_force = e["committed"]["warrant_verdict"]["predictions"]["predicted_force_n"]
            act_force = e["execution_outcome"]["actual_force_n"]
            force_errors.append((pred_force - act_force) ** 2)
            
    avg_force_variance = round(sum(force_errors) / len(force_errors), 4) if force_errors else 0.0

    print("\n========== POST-HOC EPISTEMIC AUDIT REPORT (Task 5.1) ==========")
    print(f"Total Proposals Evaluated : {total_entries}")
    print(f" -> Approved Executions   : {total_approved}")
    print(f" -> Intercepted / Rejected: {total_rejected}")
    print(f"Most Frequent Rejection Criterion : '{most_frequent_rejection}' (Count: {rejection_counts.get(most_frequent_rejection, 0)})")
    print(f"Full Rejection Breakdown          : {rejection_counts}")
    print(f"Average Force Prediction Variance : {avg_force_variance} N^2")
    print("================================================================")
    
    return {
        "total_proposals": total_entries,
        "approved": total_approved,
        "rejected": total_rejected,
        "most_frequent_rejection": most_frequent_rejection,
        "rejection_counts": rejection_counts,
        "average_force_variance": avg_force_variance
    }

# 1. Execute a safe, approved proposal so the log contains both rejections and an executed run
belief_state.battery_level_pct = 100.0  # Fully recharged
p_safe = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.22, tilt_rate_radps=0.02)
w_safe = warrant_gate.validate(p_safe, goal, belief_state)

print("Evaluating Safe Proposal: Passed =", w_safe["passed"])
if w_safe["passed"]:
    final_outcome = executive.execute_action(p_safe, goal, w_safe)
    executive.record_trace("Bring me my coffee fast!", goal, belief_state, p_safe, w_safe, raw_obs, committed=True, execution_outcome=final_outcome)
    print(f"Execution Outcome: {final_outcome}")

# 2. Run the audit analyzer on the accumulated audit trail
metrics = audit_analyzer(executive.audit_trail)


Evaluating Safe Proposal: Passed = True

[ACTUATOR] Executing motion in physical world: v_max=0.22 m/s, tilt=0.02 rad/s
Execution Outcome: {'actual_force_n': 1.91, 'actual_wave_mm': 0.71, 'actual_volume_transferred_ml': 12.0, 'spill_occurred': False, 'status': 'SUCCESS'}

========== POST-HOC EPISTEMIC AUDIT REPORT (Task 5.1) ==========
Total Proposals Evaluated : 4
 -> Approved Executions   : 1
 -> Intercepted / Rejected: 3
Most Frequent Rejection Criterion : 'safe' (Count: 2)
Full Rejection Breakdown          : {'safe': 2, 'energetically_viable': 1}
Average Force Prediction Variance : 0.0121 N^2


## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

### Answers

#### 1. Key Parts Identified from the Diagram (`robot-body.jpg`)
The image shows a PR2-style mobile service robot holding a tray. The callouts highlight its main mechanical and sensing components:

- **Blue callouts (Body & Actuation):**
  - **Arm (Links & Joints):** The articulated robotic arm made of rigid structural segments (links) connected by motorized pivot points (joints), providing the reach needed to grasp objects on counters.
  - **Gripper (End-Effector):** Two-finger parallel-jaw gripper at the end of the wrist, holding the tray and mugs.
  - **Actuator (Motor):** An electric brushless DC motor with gearbox and encoder, providing the torque to rotate arm joints and drive wheels.
  - **Wheeled Base:** Caster wheel drive modules under the base allowing the robot to navigate across flat floors.

- **Red callouts (Sensing & Perception):**
  - **Head Camera (RGB-D 3D Vision):** The sensor bar on top of the head that captures color images and 3D depth point clouds (the inset shows the robot's view of a kitchen counter with a stove and pans).
  - **Base Laser Scanner (LiDAR):** A planar laser scanner at the front of the base measuring distances to obstacles (shown by the red line measuring $1.42\text{ m}$ to the table).

---

#### 2. Core Concepts Explained
- **Links vs. Joints:** Links are the rigid structural members ("bones") that do not bend and maintain fixed lengths. Joints are the movable connections between links ("articulations") that provide rotational (revolute) or sliding (prismatic) degrees of freedom.
- **Actuators vs. Sensors:** Actuators are output devices ("muscles", like motors) that consume power to exert forces and move the robot. Sensors are input devices ("senses", like cameras, encoders, and laser scanners) that measure physical properties from the environment or body.
- **Effectors vs. Bases vs. Torsos vs. Heads:**
  - *Effector:* The terminal tool (gripper) that directly interacts with objects in the world.
  - *Base:* The bottom mobile platform that moves the entire system between rooms.
  - *Torso:* The central body that houses computers, batteries, and the vertical lift spine.
  - *Head:* The pan-tilt sensor mast that points cameras at areas of interest without needing to turn the heavy base.
- **Wheels vs. Legs vs. Grippers:** Wheels provide smooth, continuous, vibration-free motion on flat indoor floors—which is ideal for carrying drinks because it avoids the jerky stepping shocks of legged robots that cause sloshing. Legs are better for stairs and rough terrain but require complex balance. Grippers are specialized contact effectors that apply controlled friction forces to hold objects securely without slipping or breaking them.
- **Proprioception vs. Exteroception vs. Interoception:**
  - *Proprioception:* Sensing the robot's own internal mechanical state (joint angles and velocities from optical encoders, motor currents).
  - *Exteroception:* Sensing the outside world (RGB-D camera depth, LiDAR distances, tactile surface contact).
  - *Interoception:* Sensing internal physiological/hardware health (battery percentage from the BMS, motor temperatures, CPU load).

---

#### 3. How These Functional Parts Support the Cognitive Architecture
- **RGB-D Head Camera (Exteroception):** Feeds raw images and depth into `SensingSystem` and `PerceptionSystem` to update `BeliefState.eva_position` and check for obstacles. Fulfills the `monitorable` criterion in the `WarrantGate`.
- **Base LiDAR (Exteroception):** Measures distance to furniture (e.g. 1.42 m), which `SemanticDigitalTwin` uses to enforce safe navigation speed limits (`max_nav_velocity_mps`).
- **Joint Encoders (Proprioception):** Provide real-time angle and velocity feedback during `ExecutionEngine.execute_action()` so the low-level controller can follow the planned trajectory accurately.
- **Battery System (Interoception):** Provides `battery_level_pct` to `BeliefState`. The `WarrantGate` checks this in `energetically_viable`, triggering a recharge redirect if power is too low.
- **Arm Joints & Motors (Actuators):** Execute the arm tilt rate ($\dot{\theta}$) and grasp trajectory from `ParameterPacket`. The `SemanticDigitalTwin` sloshing model constrains this motion to keep wave height below the cup rim.
- **Parallel-Jaw Gripper (End-Effector):** Executes the grasp pose. Our dual-stage contact model constrains the grip force to stay below the $4.5\text{ N}$ limit to prevent cracking Eva's mug.
- **Wheeled Base (Actuators):** Drives at velocity $v_{\text{max}}$ with smooth acceleration curves so inertial forces do not cause liquid to spill over the rim.


# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 